In [10]:
import sqlite3
from pathlib import Path
from subprocess import run

from crossfiledialog import choose_folder

from src.ss import SS

### **📂 Open a (empty) folder as collection**

In [2]:
collection_path = Path(choose_folder())
collection_path

PosixPath('/home/fowel/Pictures/Collection')

### **🖼️ Add some images if needed**

In [3]:
# Select a folder, import all images within it
image_folder_path = Path(choose_folder())
run(["target/release/cli", str(collection_path), "add", "folder", str(image_folder_path)], check=False)

Generating thumbnail for /home/fowel/Pictures/Collection/images/252b290ac7b557fc1099a860cac0c5738ac0521086ba246ae442b99df4d59ac0/img-0.jpg
Generating thumbnail for /home/fowel/Pictures/Collection/images/90c960ab2d6135ba6fca20440ae7eb53f36c468723bf48fa69cba40271c13eec/img-1.jpg
Generating thumbnail for /home/fowel/Pictures/Collection/images/9ebb25ff04840ef216174a40d7f2a4c1a685a253538c602d7be12e121b2e13cb/img-2.jpg
Generating thumbnail for /home/fowel/Pictures/Collection/images/ada21d92bd07c446292786c30fbd1a9698d78056801fcb38f66a16824bd14124/img-3.jpg

Take 2.1138053 seconds



CompletedProcess(args=['target/release/cli', '/home/fowel/Pictures/Collection', 'add', 'folder', '/mnt/LOCAL/Resources/Fixtures/xs'], returncode=0)

### **🍵 Prepare semantic search and database**

In [11]:
ss = SS(collection_path / "embeddings")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [5]:
index_db = sqlite3.connect(collection_path / "index.db")
index_db.row_factory = sqlite3.Row

### **Indexing**

In [6]:
images = index_db.execute("SELECT id, hash, fullname FROM images").fetchall()
images = [{
    "id": str(img["id"]),
    "path": collection_path / "images" / img["hash"] / img["fullname"],
} for img in images]
images

[{'id': '1',
  'path': PosixPath('/home/fowel/Pictures/Collection/images/96c8647cae40e951acbf272e7c1c7025a7f74b9f59bde926991acadc9f26b627/champon-g31fa88e14_640.jpg')},
 {'id': '2',
  'path': PosixPath('/home/fowel/Pictures/Collection/images/0c0eaf8b09cdd840c840cfd1b221332b2e3979eb44a11b5b6acbc240d75d12ce/chicken-soup-g76231cf36_640.jpg')},
 {'id': '3',
  'path': PosixPath('/home/fowel/Pictures/Collection/images/3fe0a2d21b409d4b95bf33bcf1e8463e877274c48fa5a7113a28e4b8c4cd99e8/child-g8f11ee379_640.jpg')},
 {'id': '4',
  'path': PosixPath('/home/fowel/Pictures/Collection/images/971a8d27623c3e8c1960f8c9180dd31fefdb4990508e657c9d9ae0c406a1b4e7/children-g13a48a038_640.jpg')},
 {'id': '5',
  'path': PosixPath('/home/fowel/Pictures/Collection/images/379cada948c7585d9b1758cac37cc935a9b4a5632639de6adafa5b82c4361cbc/family-gc23518eae_640.jpg')},
 {'id': '6',
  'path': PosixPath('/home/fowel/Pictures/Collection/images/e8e1d44ee50f6eec6b699d6841f43da43bf0f71bfaf6bede05cf334a50d515b0/family-gcf1c70

In [12]:
ss.index_images(images)

32/85 images indexed
64/85 images indexed
85/85 images indexed


### **Search**

In [8]:
from pprint import pp

queries = [
    "Airplane",
    "A picture of the moon",
    "Thunder storm",
    "Flowers",
    "A picture of forest",
    "Leaf",
]

for query in queries:
    print(query)
    pp(ss.search(query, limit=3))
    print()

Airplane
[]

A picture of the moon
[]

Thunder storm
[]

Flowers
[]

A picture of forest
[]

Leaf
[]

